# Brain Surface Visualization: ISPC and IS-RSA Results

Visualizes parcel-wise neuroimaging results on inflated brain surfaces (Schaefer 400 + Tian S3 atlas).

| Figure | Content |
|--------|--------|
| 1 | ISPC — 4 condition maps, significant parcels (FDR < 0.05) |
| 2 | ISPC — intersection: significant in ≥ 1 condition |
| 3 | ISPC — intersection: significant in ALL 4 conditions |
| 4 | IS-RSA — 12 maps (3 measures × 4 conditions) |
| 5 | ISPC subcortical — volume-space plots for significant subcortical parcels |

**Strategy**: volumetric atlas → per-parcel NIfTI stat map → `vol_to_surf` projection → `plot_surf_stat_map`

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use('Agg')  # non-interactive backend; remove if using Jupyter with display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec
from nilearn import datasets, surface, plotting
import io
import os
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Assistant', 'Arial', 'Helvetica', 'DejaVu Sans']

print('nilearn:', __import__('nilearn').__version__)
print('nibabel:', nib.__version__)
print('numpy:', np.__version__)

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR  = '/path/to/project'
ATLAS_DIR = os.path.join(BASE_DIR, 'data/atlases')
DERIV_DIR = os.path.join(BASE_DIR, 'data/derivatives')

ATLAS_NII    = os.path.join(ATLAS_DIR, 'Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz')
ATLAS_LABELS = os.path.join(ATLAS_DIR, 'Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv')

ISPC_CSV  = os.path.join(DERIV_DIR, 'parcel_ispc/leftwing/parcel_isc_B_significance.csv')

ISRSA_WARMTH_DIR = os.path.join(DERIV_DIR, 'rsa/parcel_isrsa_leftwing/warmth_isrsa')
ISRSA_POLAR_DIR  = os.path.join(DERIV_DIR, 'rsa/parcel_isrsa_leftwing/all_conditions')

OUTPUT_DIR = os.path.join(DERIV_DIR, 'surface_brain_maps_publication')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── IS-RSA file map ────────────────────────────────────────────────────────────
CONDITIONS = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight']
COND_LABELS = {'AntiLeft': 'Anti-Left', 'AntiRight': 'Anti-Right',
                'ProLeft': 'Pro-Left',   'ProRight': 'Pro-Right'}
COND_COLORS = {'AntiLeft': 'steelblue', 'AntiRight': 'crimson',
                'ProLeft':  'forestgreen', 'ProRight': 'darkorange'}

ISRSA_FILES = {
    'Warmth\n(in-party)': {
        c: os.path.join(ISRSA_WARMTH_DIR, f'B_{c}_inparty.csv') for c in CONDITIONS
    },
    'Warmth\n(out-party)': {
        c: os.path.join(ISRSA_WARMTH_DIR, f'B_{c}_outparty.csv') for c in CONDITIONS
    },
    'Polarization': {
        c: os.path.join(ISRSA_POLAR_DIR, f'approachB_{c}.csv') for c in CONDITIONS
    },
}

# ── Surface resolution ─────────────────────────────────────────────────────────
# 'fsaverage5' (~10 k vertices, fast) vs 'fsaverage' (~160 k, publication quality)
SURF_RES = 'fsaverage5'
PANEL_DPI = 150   # DPI for individual surface renders
FIGURE_DPI = 300  # DPI for saved figures

print(f'Output → {OUTPUT_DIR}')

In [ ]:
# ── Load atlas and surfaces ────────────────────────────────────────────────────
def load_atlas(atlas_nii_path, atlas_labels_path):
    """Load volumetric atlas + label ↔ ID mappings."""
    atlas_img = nib.load(atlas_nii_path)
    labels_df = pd.read_csv(atlas_labels_path, sep='\t')
    id_to_name = dict(zip(labels_df['id'], labels_df['name']))
    name_to_id = dict(zip(labels_df['name'], labels_df['id']))
    print(f'Atlas: {atlas_img.shape}, {len(labels_df)} parcels '
          f'(cortical 1–400, subcortical 401–{labels_df["id"].max()})')
    return atlas_img, id_to_name, name_to_id

atlas_img, id_to_name, name_to_id = load_atlas(ATLAS_NII, ATLAS_LABELS)
fsaverage = datasets.fetch_surf_fsaverage(SURF_RES)
print(f'Surfaces loaded ({SURF_RES})')

In [ ]:
# ── Data exploration ───────────────────────────────────────────────────────────
df_ispc = pd.read_csv(ISPC_CSV)
print('ISPC significance CSV columns:', df_ispc.columns.tolist())
print()
for cond in CONDITIONS:
    sub = df_ispc[df_ispc['condition'] == cond]
    n_sig = sub['significant'].sum()
    n_sub_sig = sub[(sub['parcel_id'] > 400) & sub['significant']].shape[0]
    print(f'  {cond}: {n_sig:3d} sig cortical+subcortical  '
          f'({n_sub_sig} subcortical)')

sig_vals = df_ispc[df_ispc['significant']]['isc_mean']
print(f'\nISPC isc_mean (significant): {sig_vals.min():.3f} – {sig_vals.max():.3f}')
ISPC_VMAX = float(np.nanpercentile(sig_vals, 98))
print(f'ISPC_VMAX (98th pct): {ISPC_VMAX:.4f}')

print('\nIS-RSA significant parcels per file:')
all_rsa_abs = []
for measure, cond_files in ISRSA_FILES.items():
    for cond, fpath in cond_files.items():
        d = pd.read_csv(fpath)
        all_rsa_abs.extend(d['rsa_r'].dropna().abs().tolist())
        n = d['significant'].sum()
        print(f'  {measure.replace(chr(10)," "):22s} {cond}: {n} sig')

ISRSA_VMAX = float(np.nanpercentile(all_rsa_abs, 99)) if all_rsa_abs else 0.3
print(f'\nIS-RSA vmax (99th pct |rsa_r|): {ISRSA_VMAX:.4f}')

In [ ]:
# ── Core utility functions ─────────────────────────────────────────────────────

def parcels_to_stat_nifti(parcel_values_dict, atlas_img, cortical_only=True):
    """
    Build a NIfTI stat map: significant parcel voxels → stat value, rest → 0.

    Parameters
    ----------
    parcel_values_dict : {int: float}
        parcel_id (1-indexed) → stat value.
    atlas_img : Nifti1Image
    cortical_only : bool
        If True, skip parcels with id > 400.
    """
    atlas_data = np.asarray(atlas_img.get_fdata(), dtype=float)
    stat_data  = np.zeros_like(atlas_data)
    for pid, val in parcel_values_dict.items():
        if cortical_only and pid > 400:
            continue
        stat_data[atlas_data == pid] = val
    return nib.Nifti1Image(stat_data, atlas_img.affine, atlas_img.header)


def stat_nifti_to_surface(stat_nifti, fsaverage, hemi='left'):
    """Project volumetric stat map to surface vertices via vol_to_surf."""
    pial = fsaverage[f'pial_{hemi}']
    return surface.vol_to_surf(
        stat_nifti, pial,
        interpolation='nearest',
        kind='line',
        n_samples=1,   # single mid-cortex sample preserves parcel boundaries
    )


def parcels_to_surface(parcel_values_dict, atlas_img, fsaverage, hemi='left'):
    """One-step: parcel value dict → surface vertex array."""
    stat_nifti = parcels_to_stat_nifti(parcel_values_dict, atlas_img)
    return stat_nifti_to_surface(stat_nifti, fsaverage, hemi)


def _get_fig(display):
    """Extract matplotlib Figure from nilearn display or figure object."""
    if isinstance(display, plt.Figure):
        return display
    if hasattr(display, 'figure'):
        return display.figure
    if hasattr(display, 'get_figure'):
        return display.get_figure()
    return display


print('Core functions defined.')

In [ ]:
# ── Surface rendering helpers ──────────────────────────────────────────────────

def render_surf_view(texture, fsaverage, hemi, view,
                     cmap, vmax, vmin=None,
                     threshold=1e-10, symmetric_cbar=False,
                     dpi=PANEL_DPI):
    """
    Render one surface view to a numpy RGBA image array.

    Parameters
    ----------
    texture : np.ndarray  (n_vertices,)
        Surface stat values; 0 = masked (hidden by threshold).
    hemi    : 'left' | 'right'
    view    : 'lateral' | 'medial' | 'dorsal' | 'ventral' | 'anterior' | 'posterior'
    cmap    : str or Colormap
    vmax    : float
    vmin    : float or None  (None → 0 for sequential, -vmax for symmetric)
    """
    surf_mesh = fsaverage[f'infl_{hemi}']
    bg_map    = fsaverage[f'sulc_{hemi}']

    disp = plotting.plot_surf_stat_map(
        surf_mesh=surf_mesh,
        stat_map=texture,
        bg_map=bg_map,
        hemi=hemi,
        view=view,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        colorbar=False,
        threshold=threshold,
        symmetric_cbar=symmetric_cbar,
        bg_on_data=True,
        darkness=0.7,
    )
    fig = _get_fig(disp)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi,
                bbox_inches='tight', facecolor='white', pad_inches=0)
    plt.close('all')
    buf.seek(0)
    return plt.imread(buf)


def plot_condition_map(ax, img_array, title=None, title_fontsize=13):
    """Embed a rendered surface image into a matplotlib Axes."""
    ax.imshow(img_array)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=title_fontsize, pad=1)


def add_colorbar(fig, cmap, vmin, vmax, label, rect=None,
                 orientation='vertical', n_ticks=5):
    """Add a standalone colorbar to a figure."""
    if rect is None:
        rect = [0.92, 0.12, 0.015, 0.76]
    cbar_ax = fig.add_axes(rect)
    sm = ScalarMappable(cmap=cmap, norm=Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation=orientation)
    cbar.set_label(label, fontsize=13)
    ticks = np.linspace(vmin, vmax, n_ticks)
    cbar.set_ticks(ticks)
    cbar.ax.tick_params(labelsize=11)
    return cbar


print('Rendering helpers defined.')

In [ ]:
# ── Load ISPC data ─────────────────────────────────────────────────────────────

def load_ispc_data(csv_path):
    """
    Returns
    -------
    ispc_data : {condition: {parcel_id: isc_mean}} (significant parcels only)
    df        : full DataFrame
    """
    df = pd.read_csv(csv_path)
    ispc_data = {}
    for cond in df['condition'].unique():
        sub = df[(df['condition'] == cond) & df['significant']]
        ispc_data[cond] = dict(zip(sub['parcel_id'].astype(int),
                                   sub['isc_mean'].astype(float)))
    return ispc_data, df


ispc_data, df_ispc = load_ispc_data(ISPC_CSV)
print('Loaded ISPC data.')
for cond, vals in ispc_data.items():
    print(f'  {cond}: {len(vals)} significant parcels')

In [ ]:
# ── Figure 1: ISPC — 4 condition maps (4 rows × 4 cols) ───────────────────────

def make_ispc_figure(ispc_data, atlas_img, fsaverage,
                     vmax=None, cmap='YlOrRd', output_path=None, dpi=300):
    """
    4 rows (conditions) × 4 columns (LH lateral, LH medial, RH lateral, RH medial).
    Colored by isc_mean; non-significant parcels are transparent.
    """
    views      = [('left',  'lateral'), ('left',  'medial'),
                  ('right', 'lateral'), ('right', 'medial')]
    col_labels = ['LH Lateral', 'LH Medial', 'RH Lateral', 'RH Medial']

    if vmax is None:
        all_v = [v for d in ispc_data.values() for v in d.values()]
        vmax = float(np.percentile(all_v, 98)) if all_v else 0.3

    n_rows, n_cols = len(CONDITIONS), len(views)
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(20, 3.8 * n_rows), facecolor='white')

    for r, cond in enumerate(CONDITIONS):
        parcel_vals = {pid: v for pid, v in ispc_data.get(cond, {}).items()
                       if pid <= 400}  # cortical only for surface plot
        stat_nifti  = parcels_to_stat_nifti(parcel_vals, atlas_img, cortical_only=True)
        tex = {
            'left':  stat_nifti_to_surface(stat_nifti, fsaverage, 'left'),
            'right': stat_nifti_to_surface(stat_nifti, fsaverage, 'right'),
        }
        for c, (hemi, view) in enumerate(views):
            ax  = axes[r, c]
            img = render_surf_view(tex[hemi], fsaverage, hemi, view,
                                   cmap=cmap, vmin=0, vmax=vmax,
                                   threshold=1e-10, symmetric_cbar=False)
            plot_condition_map(ax, img, title=col_labels[c] if r == 0 else None,
                               title_fontsize=13)

    # set_ylabel is hidden by axis('off') — use fig.text for row labels
    plt.subplots_adjust(left=0.08, right=0.90, hspace=0.01, wspace=0.01)
    for r, cond in enumerate(CONDITIONS):
        pos = axes[r, 0].get_position()
        fig.text(0.03, pos.y0 + pos.height / 2,
                 COND_LABELS[cond],
                 va='center', ha='center', fontsize=14, fontweight='bold',
                 rotation=90)

    add_colorbar(fig, cmap, vmin=0, vmax=vmax, label='ISC mean (r)',
                 rect=[0.91, 0.12, 0.015, 0.76])
    fig.suptitle('Inter-Subject Phase Concordance (ISPC)\n'
                 'Significant cortical parcels  (FDR < 0.05)',
                 fontsize=16, fontweight='bold', y=1.01)

    if output_path:
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f'Saved → {output_path}')
    return fig


print('make_ispc_figure defined.')

In [ ]:
# ── Figures 2a/2b: ISPC pairwise intersection — agreed vs disagreed content ────
#
# Agreed content   : Pro-Left × Anti-Right (cross-party agreement on left content)
# Disagreed content: Anti-Left × Pro-Right (cross-party disagreement)
#
# Each figure: 3 categories
#   value 1 → only condition A  (color_a)
#   value 2 → only condition B  (color_b)
#   value 3 → both conditions   (color_both = gold)

def make_intersection_pair(ispc_data, atlas_img, fsaverage,
                            cond_a, cond_b,
                            color_a, color_b, color_both,
                            label_a, label_b,
                            title, output_path=None, dpi=300):
    """
    Single-row 4-view map for two-condition intersection.
    Categorical coloring: only-A, only-B, and shared parcels.
    """
    parcels_a = set(pid for pid in ispc_data.get(cond_a, {}) if pid <= 400)
    parcels_b = set(pid for pid in ispc_data.get(cond_b, {}) if pid <= 400)

    parcel_cat = {}
    for pid in parcels_a - parcels_b:
        parcel_cat[pid] = 1  # only A
    for pid in parcels_b - parcels_a:
        parcel_cat[pid] = 2  # only B
    for pid in parcels_a & parcels_b:
        parcel_cat[pid] = 3  # both

    cat_cmap = ListedColormap([color_a, color_b, color_both])

    views      = [('left',  'lateral'), ('left',  'medial'),
                  ('right', 'lateral'), ('right', 'medial')]
    col_labels = ['LH Lateral', 'LH Medial', 'RH Lateral', 'RH Medial']

    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), facecolor='white')

    stat_nifti = parcels_to_stat_nifti(parcel_cat, atlas_img, cortical_only=True)
    tex = {
        'left':  stat_nifti_to_surface(stat_nifti, fsaverage, 'left'),
        'right': stat_nifti_to_surface(stat_nifti, fsaverage, 'right'),
    }
    for c, (hemi, view) in enumerate(views):
        img = render_surf_view(tex[hemi], fsaverage, hemi, view,
                               cmap=cat_cmap, vmin=0.5, vmax=3.5,
                               threshold=0.5, symmetric_cbar=False)
        plot_condition_map(axes[c], img, title=col_labels[c], title_fontsize=14)

    n_a    = len(parcels_a - parcels_b)
    n_b    = len(parcels_b - parcels_a)
    n_both = len(parcels_a & parcels_b)

    legend_els = [
        mpatches.Patch(color=color_a,    label=f'{label_a} only  (n={n_a})'),
        mpatches.Patch(color=color_b,    label=f'{label_b} only  (n={n_b})'),
        mpatches.Patch(color=color_both, label=f'Both  (n={n_both})'),
    ]
    fig.legend(handles=legend_els, loc='lower center', ncol=3,
               fontsize=12, frameon=True, bbox_to_anchor=(0.5, -0.04))

    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()

    if output_path:
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f'Saved → {output_path}')
    return fig


print('make_intersection_pair defined.')

In [ ]:
# ── Figure 3: ISPC intersection — significant in ALL 4 conditions ──────────────

def make_intersection_all(ispc_data, atlas_img, fsaverage,
                           output_path=None, dpi=300):
    """
    Single-row, 4-view map.  Color = mean isc_mean across all 4 conditions.
    """
    # Parcels significant in all 4 conditions (cortical only)
    common = set(pid for pid in ispc_data[CONDITIONS[0]] if pid <= 400)
    for cond in CONDITIONS[1:]:
        common &= set(ispc_data[cond].keys())

    print(f'Parcels significant in ALL 4 conditions: {len(common)}')
    if not common:
        print('  → No parcels; skipping Figure 3.')
        return None

    parcel_vals = {pid: float(np.mean([ispc_data[c][pid] for c in CONDITIONS]))
                   for pid in common}
    vmax = max(parcel_vals.values())
    vmax = max(vmax, 0.01)

    views      = [('left',  'lateral'), ('left',  'medial'),
                  ('right', 'lateral'), ('right', 'medial')]
    col_labels = ['LH Lateral', 'LH Medial', 'RH Lateral', 'RH Medial']

    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), facecolor='white')

    stat_nifti = parcels_to_stat_nifti(parcel_vals, atlas_img, cortical_only=True)
    tex = {
        'left':  stat_nifti_to_surface(stat_nifti, fsaverage, 'left'),
        'right': stat_nifti_to_surface(stat_nifti, fsaverage, 'right'),
    }
    for c, (hemi, view) in enumerate(views):
        img = render_surf_view(tex[hemi], fsaverage, hemi, view,
                               cmap='YlOrRd', vmin=0, vmax=vmax,
                               threshold=1e-10, symmetric_cbar=False)
        plot_condition_map(axes[c], img, title=col_labels[c], title_fontsize=14)

    plt.subplots_adjust(right=0.90)
    add_colorbar(fig, 'YlOrRd', vmin=0, vmax=vmax, label='Mean ISC (r)',
                 rect=[0.91, 0.12, 0.015, 0.76])

    fig.suptitle(f'ISPC Intersection — Significant in ALL 4 Conditions (FDR < 0.05)\n'
                 f'N = {len(common)} cortical parcels, color = mean ISC',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout(rect=[0, 0, 0.90, 1])

    if output_path:
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f'Saved → {output_path}')
    return fig


print('make_intersection_all defined.')

In [ ]:
# ── IS-RSA data loading ────────────────────────────────────────────────────────

def load_isrsa_file(csv_path, name_to_id):
    """
    Returns
    -------
    parcel_vals : {int parcel_id: float rsa_r}  — significant parcels only
    df          : full DataFrame
    """
    df  = pd.read_csv(csv_path)
    sig = df[df['significant']]
    parcel_vals = {}
    for _, row in sig.iterrows():
        pid = name_to_id.get(row['parcel'])
        if pid is not None:
            parcel_vals[int(pid)] = float(row['rsa_r'])
    return parcel_vals, df


print('load_isrsa_file defined.')

In [ ]:
# ── Figure 4: IS-RSA — 3 measures × 4 conditions ──────────────────────────────

def make_isrsa_figure(isrsa_files, atlas_img, fsaverage, name_to_id,
                       vmax=None, cmap='RdBu_r', output_path=None, dpi=300):
    """
    Layout: 3 rows (behavioral measures) × 4 columns (conditions).
    Each cell shows LH lateral | RH lateral side-by-side.
    """
    measures = list(isrsa_files.keys())

    if vmax is None:
        abs_vals = []
        for m_files in isrsa_files.values():
            for fp in m_files.values():
                d = pd.read_csv(fp)
                abs_vals.extend(d['rsa_r'].dropna().abs().tolist())
        vmax = float(np.nanpercentile(abs_vals, 99)) if abs_vals else 0.3

    # Guard against NaN/zero vmax (can cause colorbar errors)
    if not np.isfinite(vmax) or vmax < 1e-6:
        vmax = 0.3

    n_rows, n_cols = len(measures), len(CONDITIONS)

    # Each cell has 2 sub-columns (LH, RH).  Use GridSpec for tight spacing.
    fig = plt.figure(figsize=(26, 4.2 * n_rows + 0.8), facecolor='white')
    gs  = GridSpec(n_rows, n_cols * 2, figure=fig,
                   hspace=0.02, wspace=0.005,
                   top=0.93, bottom=0.04, left=0.06, right=0.90)

    for r, measure in enumerate(measures):
        for ci, cond in enumerate(CONDITIONS):
            fp = isrsa_files[measure][cond]
            parcel_vals, _ = load_isrsa_file(fp, name_to_id)

            stat_nifti = parcels_to_stat_nifti(
                {pid: v for pid, v in parcel_vals.items() if pid <= 400},
                atlas_img, cortical_only=True
            )
            tex_lh = stat_nifti_to_surface(stat_nifti, fsaverage, 'left')
            tex_rh = stat_nifti_to_surface(stat_nifti, fsaverage, 'right')

            col_base = ci * 2  # two grid columns per condition

            ax_lh = fig.add_subplot(gs[r, col_base])
            img_lh = render_surf_view(tex_lh, fsaverage, 'left', 'lateral',
                                      cmap=cmap, vmin=-vmax, vmax=vmax,
                                      threshold=1e-10, symmetric_cbar=True)
            plot_condition_map(ax_lh, img_lh)

            ax_rh = fig.add_subplot(gs[r, col_base + 1])
            img_rh = render_surf_view(tex_rh, fsaverage, 'right', 'lateral',
                                      cmap=cmap, vmin=-vmax, vmax=vmax,
                                      threshold=1e-10, symmetric_cbar=True)
            plot_condition_map(ax_rh, img_rh)

            # Column header (first row only)
            if r == 0:
                ax_lh.set_title(COND_LABELS[cond], fontsize=13,
                                fontweight='bold', pad=4)

    # Row labels
    for r, measure in enumerate(measures):
        ax0 = fig.add_subplot(gs[r, 0])
        pos = ax0.get_position()
        ax0.set_visible(False)
        fig.text(0.01, pos.y0 + pos.height / 2,
                 measure,
                 va='center', ha='left', fontsize=13, fontweight='bold',
                 rotation=90)

    # Shared colorbar
    add_colorbar(fig, cmap, vmin=-vmax, vmax=vmax, label='RSA r',
                 rect=[0.91, 0.10, 0.015, 0.80])

    fig.suptitle('Inter-Subject RSA (IS-RSA) — Lateral views\n'
                 'Significant cortical parcels (FDR < 0.05)',
                 fontsize=16, fontweight='bold')

    if output_path:
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f'Saved → {output_path}')
    return fig


print('make_isrsa_figure defined.')

In [ ]:
# ── Figure 5: Subcortical ISPC in volume space ────────────────────────────────

def make_subcortical_figure(ispc_data, atlas_img, output_path=None, dpi=300):
    """
    For each condition, plot significant subcortical parcels (id > 400)
    using nilearn's plot_stat_map (MNI volume space).
    Layout: 4 rows (conditions) × 1 column of axial slices.
    """
    fig, axes = plt.subplots(len(CONDITIONS), 1,
                              figsize=(16, 3.8 * len(CONDITIONS)), facecolor='white')

    any_subcortical = False
    for r, cond in enumerate(CONDITIONS):
        sub_vals = {pid: v for pid, v in ispc_data.get(cond, {}).items() if pid > 400}
        if not sub_vals:
            axes[r].set_visible(False)
            continue
        any_subcortical = True

        stat_nifti = parcels_to_stat_nifti(sub_vals, atlas_img, cortical_only=False)

        # Use the entire atlas as background context
        vmax_sub = max(sub_vals.values())
        disp = plotting.plot_stat_map(
            stat_nifti,
            display_mode='z',
            cut_coords=6,
            colorbar=True,
            cmap='YlOrRd',
            vmax=vmax_sub,
            threshold=1e-10,
            title=f'{COND_LABELS[cond]}  (subcortical, FDR < 0.05)',
            axes=axes[r],
        )

    if not any_subcortical:
        print('No significant subcortical parcels — skipping Figure 5.')
        plt.close(fig)
        return None

    fig.suptitle('ISPC — Significant Subcortical Parcels (volume space)',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()

    if output_path:
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f'Saved → {output_path}')
    return fig


print('make_subcortical_figure defined.')

In [ ]:
# ── Run all figures ────────────────────────────────────────────────────────────

print('=' * 65)
print('Figure 1: ISPC — 4 conditions …')
fig1 = make_ispc_figure(
    ispc_data, atlas_img, fsaverage,
    vmax=ISPC_VMAX, cmap='YlOrRd',
    output_path=os.path.join(OUTPUT_DIR, 'ispc_4conditions.png'),
    dpi=FIGURE_DPI,
)
plt.show()

print('\nFigure 2a: ISPC — agreed content (Pro-Left × Anti-Right) …')
fig2a = make_intersection_pair(
    ispc_data, atlas_img, fsaverage,
    cond_a='ProLeft',   cond_b='AntiRight',
    color_a='forestgreen', color_b='crimson', color_both='gold',
    label_a='Pro-Left', label_b='Anti-Right',
    title='ISPC Intersection — Agreed Content (FDR < 0.05)\nPro-Left  ×  Anti-Right',
    output_path=os.path.join(OUTPUT_DIR, 'ispc_intersection_agreed.png'),
    dpi=FIGURE_DPI,
)
plt.show()

print('\nFigure 2b: ISPC — disagreed content (Anti-Left × Pro-Right) …')
fig2b = make_intersection_pair(
    ispc_data, atlas_img, fsaverage,
    cond_a='AntiLeft',  cond_b='ProRight',
    color_a='steelblue', color_b='darkorange', color_both='gold',
    label_a='Anti-Left', label_b='Pro-Right',
    title='ISPC Intersection — Disagreed Content (FDR < 0.05)\nAnti-Left  ×  Pro-Right',
    output_path=os.path.join(OUTPUT_DIR, 'ispc_intersection_disagreed.png'),
    dpi=FIGURE_DPI,
)
plt.show()

print('\nFigure 3: ISPC — intersection ALL …')
fig3 = make_intersection_all(
    ispc_data, atlas_img, fsaverage,
    output_path=os.path.join(OUTPUT_DIR, 'ispc_intersection_all.png'),
    dpi=FIGURE_DPI,
)
plt.show()

print('\nFigure 4: IS-RSA — 12 maps …')
fig4 = make_isrsa_figure(
    ISRSA_FILES, atlas_img, fsaverage, name_to_id,
    vmax=ISRSA_VMAX, cmap='RdBu_r',
    output_path=os.path.join(OUTPUT_DIR, 'isrsa_12maps.png'),
    dpi=FIGURE_DPI,
)
plt.show()

print('\nFigure 5: ISPC subcortical (volume space) …')
fig5 = make_subcortical_figure(
    ispc_data, atlas_img,
    output_path=os.path.join(OUTPUT_DIR, 'ispc_subcortical_volume.png'),
    dpi=FIGURE_DPI,
)
plt.show()

print('\n' + '=' * 65)
print(f'All figures saved to:\n  {OUTPUT_DIR}')

## Notes

### Switching to full-resolution surfaces
Change `SURF_RES = 'fsaverage5'` → `'fsaverage'` in the **Configuration** cell.  
Rendering time will increase ~16× per panel; quality improves for publication.

### Threshold
- ISPC and IS-RSA surface maps: `threshold=1e-10` hides voxels where |stat| = 0 (non-significant parcels).
- Intersection-any map: `threshold=0.5` hides the background (category 0).

### Parcel ↔ vertex mapping
Strategy: volumetric atlas voxels → NIfTI stat image → `nilearn.surface.vol_to_surf`  
(nearest-neighbour, single mid-cortical depth sample) → vertex texture.  
Subcortical parcels (id > 400) are excluded from surface plots and visualised separately in MNI volume space.

### IS-RSA figure shows only lateral views
To add medial views, extend the `views` list in `make_isrsa_figure` and  
expand the GridSpec to `n_cols * 4` columns (LH lat, LH med, RH lat, RH med per condition).